# BFMN — Rassereno

Building of **behavioural forma mentis networks (BFMN)** for every sample, extraction of
their network features and export of the aggregated results.

In [ ]:
import pandas as pd
import pickle

from config import RESSERENO_ROOT, KEYWORDS_RASSERENO
from bfmn_building import (
    categorize_valence_multiple,
    pipeline_df_building,
    build_multiple_fmnt,
)

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

### Function defaults

`build_multiple_fmnt`:
- `flower="yes"` &nbsp;&rarr;&nbsp; plot the emotional flowers (set `"no"` to skip them)
- `language="ita"` &nbsp;&rarr;&nbsp; language EmoAtlas works in (set `"eng"` for English)

`pipeline_df_building`:
- `filter="no"` &nbsp;&rarr;&nbsp; keep every association (set `"yes"` to drop idiosyncratic ones, i.e. associations made by a single participant)

## Files reading

In [ ]:
def read_formatting(rel_path):
    return pd.read_csv(RESSERENO_ROOT / "out" / "formatting" / rel_path)


def read_split(rel_path):
    return pd.read_csv(RESSERENO_ROOT / "out" / "splitted_dfs" / rel_path)


all_qualtrics = read_formatting(
    "all_students_2025/associations_valence_task_qualtrics.csv"
)
high_school_south = all_qualtrics[
    all_qualtrics["participant_id"].str.startswith("high_school_south")
]

# anxiety-split files: sample name -> splitted_dfs subfolder (psychology: only the students whose
# cue set had all four target words; GPT-oss: the personas matched to those humans)
SPLIT_FOLDERS = {
    "high_school_2025": "high_school_2025",
    "physics_2025": "physics_2025",
    "psychology": "psychology_sets_4_5",
    "gpt_high_school": "high_school_gpt_oss",
    "gpt_physics": "physics_gpt_oss",
    "gpt_psychology": "psychology_gpt_oss",
}
SPLITS = {
    f"{sample}_{anx}_anx": read_split(f"{folder}/{anx}_anx.csv")
    for sample, folder in SPLIT_FOLDERS.items()
    for anx in ["high", "low"]
}


## Dataframe preparation

Every sample goes through the full `pipeline_df_building` (text cleaning &rarr; long format
&rarr; valence categorization). Every sample (human and GPT-oss) is prepared twice: **unfiltered** (network features,
emotional flowers, exports) and **filtered** (semantic-frame plots only, keeping associations
given by at least two participants for the same cue word).

In [ ]:
# every sample (human and GPT-oss) is prepared both filtered (plots) and unfiltered (analysis),
# with the same rule for all
sample_dfs = {
    f"{name}_{version}": pipeline_df_building(df, filter=flag)
    for name, df in SPLITS.items()
    for version, flag in [("filtered", "yes"), ("unfiltered", "no")]
}


In [ ]:
df_high_school_south = pipeline_df_building(high_school_south)

In [ ]:
fmnts, features_keyword, features_tot = build_multiple_fmnt(
    df_high_school_south, ["matematica", "fisica"], RESSERENO_ROOT, "high_school_south"
)

## BFMN building

Each sample is described by `(name, dataframe, keywords, output subfolder, extra kwargs)` and
built through the same `build_multiple_fmnt` call. Results are collected in `bfmn_results`, keyed
by sample name, each holding the keyword formae mentis, the per-keyword features and the whole-network
features.

In [ ]:
# (name, dataframe, keywords, output subfolder, extra build_multiple_fmnt kwargs)
# output subfolders: "{folder}_filtered/..." holds the frame plots for the figures,
# "{folder}_unfiltered/..." the unfiltered plots and the emotional flowers
OUTPUT_FOLDERS = {
    "high_school_2025": "liceo_2025",
    "physics_2025": "physics_students_2025",
    "psychology": "psychology_students",
    "gpt_high_school": "gpt_oss_high_school",
    "gpt_physics": "gpt_oss_physics",
    "gpt_psychology": "gpt_oss_psychology",
}
BFMN_BUILDS = [
    (
        f"{sample}_{anx}_anx_{version}",
        sample_dfs[f"{sample}_{anx}_anx_{version}"],
        KEYWORDS_RASSERENO,
        f"{folder}_{version}/{anx}_anxiety",
        {"flower": "yes" if version == "unfiltered" else "no"},
    )
    for sample, folder in OUTPUT_FOLDERS.items()
    for anx in ["high", "low"]
    for version in ["filtered", "unfiltered"]
]

bfmn_results = {}
for name, df, keywords, file_path, kwargs in BFMN_BUILDS:
    fmnts, features_keyword, features_tot = build_multiple_fmnt(
        df, keywords, RESSERENO_ROOT, file_path, **kwargs
    )
    bfmn_results[name] = {
        "fmnts": fmnts,
        "features_keyword": features_keyword,
        "features_tot": features_tot,
    }


## BFMN exports

- `features_keyword_fmns.csv` / `features_tot_fmns.csv`: per-keyword and whole-network features for
  every sample (human samples in their **unfiltered** version).
- `all_bfmns.pkl`: the edge lists of every keyword forma mentis (human samples in their **unfiltered**
  version, so the Jaccard analysis uses the same networks as the features).

In [ ]:
analysis_dir = RESSERENO_ROOT / "out" / "analysis"

# samples (and order) feeding the exported feature tables, all in their unfiltered version
FEATURE_EXPORT_ORDER = [
    f"{sample}_{anx}_anx_unfiltered"
    for sample in OUTPUT_FOLDERS
    for anx in ["high", "low"]
]

for feature_key, filename in [
    ("features_keyword", "features_keyword_fmns.csv"),
    ("features_tot", "features_tot_fmns.csv"),
]:
    combined = pd.concat(
        [bfmn_results[name][feature_key] for name in FEATURE_EXPORT_ORDER],
        ignore_index=True,
    )
    combined.to_csv(analysis_dir / filename, index=False)


In [ ]:
# export key -> sample name (unfiltered networks, so the Jaccard analysis uses the same networks
# as the features)
EXPORT_NAMES = {
    "high_school_2025": "high_school",
    "physics_2025": "physics",
    "psychology": "psychology",
    "gpt_high_school": "gpt_oss_high_school",
    "gpt_physics": "gpt_oss_physics",
    "gpt_psychology": "gpt_oss_psychology",
}
BFMN_EXPORT = {
    f"{export}_{anx}_anx": f"{sample}_{anx}_anx_unfiltered"
    for sample, export in EXPORT_NAMES.items()
    for anx in ["high", "low"]
}

all_bfmns = {
    export_key: [fmnt.edges for fmnt in bfmn_results[name]["fmnts"]]
    for export_key, name in BFMN_EXPORT.items()
}

with open(analysis_dir / "all_bfmns.pkl", "wb") as f:
    pickle.dump(all_bfmns, f)
